# Part C: Ticket Triage

**Goal:** Classify support tickets and route them.

**Key techniques:** NLP classification with an LLM, PII redaction, zero-shot vs few-shot prompting using past tickets as examples.

## How to use this notebook
1. Keep this notebook in the **project root** (next to `generate_data.py`, `src/`, `data/`). It also works from a sub-folder; the setup cell finds the root.
2. Select the project's `.venv` as the kernel (top-right of the notebook in VS Code).
3. Run the cells **top to bottom** (`Run All`). Later cells depend on earlier ones.

## What is inside
The code in each section is the code from your `.py` file, split into cells with no change to the logic. Sections run in this order:

1. `ticket_triage.py` - PII redaction, prompts, classification runner
2. `build_triage_outputs.py` - run zero-shot (budget demo) and few-shot
3. `evaluate_triage.py` - accuracy and confusion matrix

Uses the local `mock_llm.py`, so no API key is needed.

The original `.py` files are untouched and are still needed: some sections import each other (for example `from policy_rag import ...`) and `mock_llm.py` is the supplied local LLM.


In [ ]:
# Setup cell: lets the original script code run unchanged inside a notebook.
# 1) finds the project root, 2) makes it the working directory (the scripts use
# relative paths like "data/raw"), 3) makes the project's own modules importable,
# 4) defines __file__ (scripts use it, notebooks do not have it).
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").is_dir() and (candidate / "mock_llm.py").exists():
        ROOT = candidate
        break

os.chdir(ROOT)
for p in (str(ROOT), str(ROOT / "src")):
    if p not in sys.path:
        sys.path.insert(0, p)

__file__ = str(ROOT / "src" / "notebook.py")
print("Project root:", ROOT)

---
## `src/ticket_triage.py`
**Why:** tickets contain emails, phones and order IDs, so they are redacted *before* going to the LLM. Then each ticket is classified as billing, delivery, technical or fraud, with output validation and retries.

**Original purpose note from the script:**

> Part C: Support ticket privacy checks and guardrails.

In [ ]:
import re
import math
import pandas as pd
INPUT_PATH = "data/raw/tickets.csv"
OUTPUT_PATH = "outputs/ticket_guardrail_check.csv"
EMAIL_PATTERN = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")
PHONE_PATTERN = re.compile(r"\b[6-9]\d{9}\b")
ORDER_ID_PATTERN = re.compile(r"ORD-\d{6}")

In [ ]:
def redact_pii(text):
    """Redact email, phone, and order ID in the required order."""
    text = "" if pd.isna(text) else str(text)

    text, emails = EMAIL_PATTERN.subn("[EMAIL]", text)
    text, phones = PHONE_PATTERN.subn("[PHONE]", text)
    text, order_ids = ORDER_ID_PATTERN.subn("[ORDER_ID]", text)

    counts = {
        "email_redactions": emails,
        "phone_redactions": phones,
        "order_id_redactions": order_ids,
    }
    return text, counts

In [ ]:
def est_tokens(text):
    """Use the supplied mock LLM's token estimator."""
    import sys
    from pathlib import Path

    project_root = str(Path(__file__).resolve().parent.parent)
    if project_root not in sys.path:
        sys.path.insert(0, project_root)

    import mock_llm
    return mock_llm.est_tokens(text)

In [ ]:
def inspect_tickets():
    df = pd.read_csv(INPUT_PATH)
    required = {
        "ticket_id", "customer_id", "order_id", "text", "true_category"
    }
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    rows = []
    total_counts = {
        "email_redactions": 0,
        "phone_redactions": 0,
        "order_id_redactions": 0,
    }

    for _, ticket in df.iterrows():
        original_text = "" if pd.isna(ticket["text"]) else str(ticket["text"])
        redacted_text, counts = redact_pii(original_text)

        for key in total_counts:
            total_counts[key] += counts[key]

        if not original_text.strip():
            status = "skipped_empty"
        elif est_tokens(original_text) > 400:
            status = "flagged_long"
        else:
            status = "ready"

        # Guardrail statuses are decided before any model call.
        rows.append({
            "ticket_id": ticket["ticket_id"],
            "status": status,
            "estimated_tokens": est_tokens(original_text),
            "redacted_text": redacted_text,
            **counts,
        })

    result = pd.DataFrame(rows)
    result.to_csv(OUTPUT_PATH, index=False)

    print("Tickets checked:", len(result))
    print("Guardrail statuses:")
    print(result["status"].value_counts().to_string())
    print("PII redaction totals:")
    for key, value in total_counts.items():
        print(f"{key}: {value}")
    print("Saved:", OUTPUT_PATH)

    # Confirm no specified PII patterns remain in the redacted text.
    for pattern in (EMAIL_PATTERN, PHONE_PATTERN, ORDER_ID_PATTERN):
        assert not result["redacted_text"].fillna("").apply(
            lambda value: bool(pattern.search(value))
        ).any(), f"Residual PII found for {pattern.pattern}"

    print("PII residual checks: PASSED")

In [ ]:
if __name__ == "__main__":
    inspect_tickets()

In [ ]:
# ---------- Classification prompts and response validation ----------

import json
import random
import time
ALLOWED_CATEGORIES = {"billing", "delivery", "technical", "fraud"}
ZERO_SHOT_SYSTEM_PROMPT = """
CLASSIFY each customer support ticket into exactly one category:
billing, delivery, technical, or fraud.

Return JSON only, with exactly these keys:
{"category": "...", "reason": "..."}

Do not include markdown or any text outside the JSON.
""".strip()
FEW_SHOT_SYSTEM_PROMPT = """
CLASSIFY each customer support ticket into exactly one category:
billing, delivery, technical, or fraud.

Return JSON only, with exactly these keys:
{"category": "...", "reason": "..."}

Do not include markdown or any text outside the JSON.

EXAMPLES:
Ticket: I was charged twice for the same purchase.
Output: {"category": "billing", "reason": "duplicate charge"}

Ticket: My package has not arrived yet.
Output: {"category": "delivery", "reason": "package not received"}

Ticket: The app freezes whenever I try to log in.
Output: {"category": "technical", "reason": "app malfunction"}

Ticket: There is an unauthorized transaction on my card.
Output: {"category": "fraud", "reason": "unauthorized transaction"}
""".strip()

In [ ]:
def validate_model_response(content):
    """
    Return (status, category, reason).

    Valid JSON with an allowed category -> classified_json.
    Otherwise, accept exactly one allowed category word -> classified_keyword.
    If neither rule succeeds -> flagged_invalid_output.
    """
    content = "" if content is None else str(content)

    try:
        parsed = json.loads(content)
        if (
            isinstance(parsed, dict)
            and isinstance(parsed.get("category"), str)
            and parsed["category"].lower() in ALLOWED_CATEGORIES
        ):
            return (
                "classified_json",
                parsed["category"].lower(),
                str(parsed.get("reason", "")),
            )
    except (json.JSONDecodeError, TypeError):
        pass

    words = re.findall(r"[a-z]+", content.lower())
    category_words = [word for word in words if word in ALLOWED_CATEGORIES]
    unique_categories = set(category_words)

    if len(unique_categories) == 1:
        return ("classified_keyword", next(iter(unique_categories)), "")

    return ("flagged_invalid_output", "", "")

In [ ]:
# ---------- Retry handling ----------

BASE_DELAY = 0.05
MAX_RETRIES = 3

In [ ]:
def call_with_retry(messages, ticket_id, retry_log):
    """
    Retry transient mock LLM failures up to 3 times.
    The initial call is followed by at most 3 retries.
    """
    import mock_llm

    for retry_number in range(MAX_RETRIES + 1):
        try:
            return mock_llm.chat(messages)
        except mock_llm.TransientLLMError:
            if retry_number == MAX_RETRIES:
                raise

            wait_seconds = (
                BASE_DELAY
                * (2 ** retry_number)
                * random.uniform(0.9, 1.15)
            )

            retry_log.append({
                "ticket_id": ticket_id,
                "retry_number": retry_number + 1,
                "wait_seconds": wait_seconds,
            })

            print(
                f"Ticket {ticket_id}: retry {retry_number + 1}, "
                f"waiting {wait_seconds:.4f}s"
            )
            time.sleep(wait_seconds)

In [ ]:
# ---------- Classification runner ----------

# ---------- Classification runner ----------

def run_classification(mode, token_budget=None, output_path=None):
    import sys
    from pathlib import Path

    project_root = Path(__file__).resolve().parent.parent
    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))

    import mock_llm

    if mode not in {"zero_shot", "few_shot"}:
        raise ValueError("mode must be 'zero_shot' or 'few_shot'")

    if token_budget is not None and token_budget < 0:
        raise ValueError("token_budget must be non-negative")

    system_prompt = (
        ZERO_SHOT_SYSTEM_PROMPT
        if mode == "zero_shot"
        else FEW_SHOT_SYSTEM_PROMPT
    )

    input_path = Path(INPUT_PATH)
    if not input_path.is_absolute():
        input_path = project_root / input_path

    df = pd.read_csv(input_path)
    mock_llm.reset()

    results = []
    retry_log = []
    total_tokens = 0

    for _, ticket in df.iterrows():
        original_text = (
            "" if pd.isna(ticket["text"]) else str(ticket["text"])
        )
        redacted_text, _ = redact_pii(original_text)

        # Guardrail 1: empty tickets never reach the model.
        if not original_text.strip():
            results.append({
                "ticket_id": ticket["ticket_id"],
                "status": "skipped_empty",
                "category": "",
                "reason": "",
                "tokens_used": 0,
            })
            continue

        # Guardrail 2: long tickets never reach the model.
        if est_tokens(original_text) > 400:
            results.append({
                "ticket_id": ticket["ticket_id"],
                "status": "flagged_long",
                "category": "",
                "reason": "",
                "tokens_used": 0,
            })
            continue

        # Budget check happens before every model call.
        if (
            token_budget is not None
            and total_tokens >= token_budget
        ):
            results.append({
                "ticket_id": ticket["ticket_id"],
                "status": "budget_exhausted",
                "category": "",
                "reason": "",
                "tokens_used": 0,
            })
            continue

        # The model receives only the redacted ticket text.
        assert redacted_text == redact_pii(original_text)[0]
        assert not EMAIL_PATTERN.search(redacted_text)
        assert not PHONE_PATTERN.search(redacted_text)
        assert not ORDER_ID_PATTERN.search(redacted_text)

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": redacted_text},
        ]

        response = call_with_retry(
            messages, ticket["ticket_id"], retry_log
        )

        content = response.choices[0].message.content
        status, category, reason = validate_model_response(content)
        used_tokens = int(response.usage.total_tokens)
        total_tokens += used_tokens

        results.append({
            "ticket_id": ticket["ticket_id"],
            "status": status,
            "category": category,
            "reason": reason,
            "tokens_used": used_tokens,
        })

    results_df = pd.DataFrame(results)

    retry_df = pd.DataFrame(
        retry_log,
        columns=["ticket_id", "retry_number", "wait_seconds"],
    )

    if output_path is None:
        output_file = project_root / "outputs" / f"triage_{mode}.csv"
    else:
        output_file = Path(output_path)
        if not output_file.is_absolute():
            output_file = project_root / output_file

    output_file.parent.mkdir(parents=True, exist_ok=True)
    results_df.to_csv(output_file, index=False)

    retry_path = project_root / "outputs" / f"retry_log_{mode}.csv"
    retry_path.parent.mkdir(parents=True, exist_ok=True)
    retry_df.to_csv(retry_path, index=False)

    eligible = results_df[
        results_df["status"].isin(
            ["classified_json", "classified_keyword"]
        )
    ]

    print(f"\nMode: {mode}")
    print("Status counts:")
    print(results_df["status"].value_counts().to_string())
    print("Classified tickets:", len(eligible))
    print("Total tokens:", int(results_df["tokens_used"].sum()))
    print(
        "Tokens per classified ticket:",
        round(eligible["tokens_used"].sum() / len(eligible), 2)
        if len(eligible) else 0,
    )
    print("Retry attempts:", len(retry_log))
    print("Saved:", output_file)
    print("Saved:", retry_path)

    return results_df

---
## `src/build_triage_outputs.py`
**How:** runs zero-shot with a small token budget (to show the budget guardrail) and few-shot with the full budget, then writes the result CSVs.

In [ ]:
from pathlib import Path
import pandas as pd
PROJECT_ROOT = Path(__file__).resolve().parent.parent
OUTPUT_DIR = PROJECT_ROOT / "outputs"

In [ ]:
def main():
    import sys

    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))

    from src.ticket_triage import run_classification

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # Part 1: Demonstrate the zero-shot token budget.
    budget_results = run_classification(
        "zero_shot",
        token_budget=6000,
        output_path=OUTPUT_DIR / "triage_budget_demo.csv",
    )

    exhausted = int(
        (budget_results["status"] == "budget_exhausted").sum()
    )
    print(f"\nBudget demo: 6000 tokens")
    print(f"Tickets stopped by budget: {exhausted}")

    if exhausted == 0:
        raise RuntimeError(
            "No tickets exhausted the budget. Try a lower demo budget."
        )

    # Part 2: Run few-shot classification with the required budget.
    run_classification(
        "few_shot",
        token_budget=100000,
        output_path=OUTPUT_DIR / "triage_results.csv",
    )

    # Enrich results with customer IDs from the original tickets.
    results = pd.read_csv(OUTPUT_DIR / "triage_results.csv")
    tickets = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "tickets.csv")
    segments = pd.read_csv(OUTPUT_DIR / "customer_segments.csv")

    results = results.merge(
        tickets[["ticket_id", "customer_id"]],
        on="ticket_id",
        how="left",
        validate="one_to_one",
    )

    results = results.merge(
        segments[["customer_id", "segment_name", "is_premium"]],
        on="customer_id",
        how="left",
        validate="many_to_one",
    )

    results["segment_name"] = results["segment_name"].fillna("")

    # Missing customers are treated as not premium.
    premium = (
        results["is_premium"]
        .map({
            True: True,
            False: False,
            "True": True,
            "False": False,
            1: True,
            0: False,
        })
        .fillna(False)
    )

    results["priority"] = "NORMAL"
    results["route"] = "support_queue"

    needs_human = results["status"].isin([
        "skipped_empty",
        "flagged_long",
        "flagged_invalid_output",
        "budget_exhausted",
    ])

    is_fraud = results["category"].eq("fraud")

    results.loc[needs_human, "route"] = "human_review"

    results.loc[
        ~needs_human & is_fraud, "route"
    ] = "security_team"

    results.loc[
        ~needs_human & is_fraud, "priority"
    ] = "HIGH"

    premium_route = ~needs_human & ~is_fraud & premium

    results.loc[
        premium_route, "route"
    ] = "premium_desk"

    results.loc[
        premium_route, "priority"
    ] = "HIGH"

    # Keep exactly the columns required by the project brief.
    columns = [
        "ticket_id",
        "customer_id",
        "status",
        "category",
        "segment_name",
        "priority",
        "route",
        "tokens_used",
    ]

    results = results[columns]

    output_path = OUTPUT_DIR / "triage_results.csv"
    results.to_csv(output_path, index=False)

    # Final validation.
    assert len(results) == 164, (
        f"Expected 164 tickets, found {len(results)}"
    )
    assert results["ticket_id"].is_unique
    assert results["route"].notna().all()
    assert set(results["route"]).issubset({
        "human_review",
        "security_team",
        "premium_desk",
        "support_queue",
    })

    print("\nFinal triage routing:")
    print(results["route"].value_counts().to_string())
    print("\nFinal priority counts:")
    print(results["priority"].value_counts().to_string())
    print(f"\nValidated {len(results)} unique tickets.")
    print(f"Saved: {output_path}")
    print(f"Saved: {OUTPUT_DIR / 'triage_budget_demo.csv'}")

In [ ]:
if __name__ == "__main__":
    main()

---
## `src/evaluate_triage.py`
**Why:** measures how good the classifier is using accuracy and a confusion matrix.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix
ROOT = Path(__file__).resolve().parents[1]
tickets = pd.read_csv(ROOT / "data" / "raw" / "tickets.csv")
for mode in ["zero_shot", "few_shot"]:
    results = pd.read_csv(ROOT / "outputs" / f"triage_{mode}.csv")

    # Compare only tickets with a valid classification.
    comparison = results.merge(
        tickets[["ticket_id", "true_category"]],
        on="ticket_id",
        how="left",
    )

    classified = comparison[
        comparison["status"].isin(
            ["classified_json", "classified_keyword"]
        )
    ].copy()

    accuracy = accuracy_score(
        classified["true_category"],
        classified["category"],
    )

    print(f"\n{'=' * 40}")
    print(f"Mode: {mode}")
    print(f"Classified tickets: {len(classified)}")
    print(f"Correct predictions: "
          f"{(classified['category'] == classified['true_category']).sum()}")
    print(f"Accuracy: {accuracy:.2%}")

    labels = ["billing", "delivery", "technical", "fraud"]
    matrix = confusion_matrix(
        classified["true_category"],
        classified["category"],
        labels=labels,
    )

    print("\nConfusion matrix")
    print("Rows = actual category; columns = predicted category")
    print(pd.DataFrame(
        matrix,
        index=labels,
        columns=labels,
    ))